# Linguae — Character-Level Language Detector

This notebook trains, evaluates and exports a **single-label, character-level language detector** for Linguae using PyTorch only: no pretrained models, no external inference APIs, no scikit-learn and no TensorFlow.

The end product is a versioned, self-contained artifact in `artifacts/linguae/` that can be loaded by ordinary Python code without rerunning training.

## Pipeline

1. Load and validate a CSV with `text` and `language` columns.
2. Report dataset quality: missing values, duplicates, class balance, character-length distribution.
3. Split deterministically into train/validation/test per language, keeping duplicate texts together.
4. Build a character vocabulary from the training split only and encode strings as character IDs.
5. Train a character-level CNN with masked max pooling, AdamW and early stopping.
6. Evaluate exactly once on the held-out test set: NumPy-only metrics, confusion matrix, confidence and length analysis.
7. Export `model.pt`, `config.json`, `vocabulary.json`, `labels.json`, `metrics.json`, `manifest.json` plus a standalone inference module and example.

## Assumptions

- The CSV has columns `text` (string to classify) and `language` (preferably an ISO 639-1 code such as `eng`).
- Every row contains one dominant language. The model performs **one prediction per string** and cannot detect mixed-language spans.
- The dataset is large enough to keep every language in the training split; the split code raises a helpful error otherwise.
- Unicode and case carry signal, so text is NFC-normalised and whitespace-collapsed, but never lowercased, transliterated or accent-stripped.

## Prerequisites

- Python environment from `packages/models` (`uv sync`), providing PyTorch, pandas, NumPy, Matplotlib and Jupyter.
- WiLI-2018 CSV at `packages/models/data/csv/wili-2018.csv`; generate it with `notebooks/wili_2018_txt_to_csv.ipynb`. Point `CONFIG.dataset_path` elsewhere (for example `data/processed/dataset.csv`) to train on another file.

## Running the notebook

- Start Jupyter from `packages/models`, `packages/models/notebooks`, or the repository root. The code locates the package root by walking up to the directory containing `pyproject.toml`, `data/` and `notebooks/`, so relative paths always resolve to `packages/models`. From an unrelated directory, set `LINGUAE_PACKAGE_ROOT=/absolute/path/to/packages/models`.
- Execute all cells, or run `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/train_language_detector.ipynb`.
- On Apple Silicon MPS, one epoch over the full 235,000-row dataset takes roughly a minute; CPU-only training is much slower. Set `CONFIG.max_rows_per_language` (for example `200`) for a fast end-to-end smoke run.

## Expected artifacts

- `artifacts/linguae/`: versioned model artifact (`model.pt`, `config.json`, `vocabulary.json`, `labels.json`, `metrics.json`, `manifest.json`, `linguae_inference.py`, `example_inference.py`, `confusion_matrix.csv`, `misclassified_examples.csv`, `training_history.csv`).
- `data/processed/labels.json`: the language↔index mappings used during training.

## Re-runs and versioning

`CONFIG.model_version` is recorded in the manifest. Exporting into a non-empty artifact directory fails unless `CONFIG.allow_overwrite = True`; bump the version or the directory instead of overwriting silently.

## 1. Configuration and reproducibility

Everything configurable lives in the single immutable `Config` dataclass below: data paths, split fractions, encoding length, CNN dimensions, optimisation settings, evaluation limits and export/versioning options.

Seeds are fixed for Python, NumPy and PyTorch so that data order, weight initialisation, dropout and batch shuffling repeat. Device kernels can still differ slightly: CUDA/MPS may not be bitwise deterministic, and MPS does not support every deterministic algorithm. Exact numerical equality is therefore only guaranteed on CPU with the same library versions.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import re
import sys
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LogNorm
from torch import nn
from torch.utils.data import DataLoader, Dataset

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.max_open_warning"] = 0


def find_package_root(start: Path | None = None) -> Path:
    '''Locate packages/models by walking up from the current directory.'''
    override = os.environ.get("LINGUAE_PACKAGE_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if (root / "pyproject.toml").is_file() and (root / "data").is_dir():
            return root
        raise FileNotFoundError(
            f"LINGUAE_PACKAGE_ROOT={override!r} does not look like packages/models."
        )
    origins = [Path(start)] if start else [Path.cwd()]
    notebook_file = globals().get("__file__")
    if notebook_file:
        origins.append(Path(str(notebook_file)).resolve().parent)
    for origin in origins:
        for candidate in (origin.resolve(), *origin.resolve().parents):
            for option in (candidate, candidate / "packages" / "models"):
                if (
                    (option / "pyproject.toml").is_file()
                    and (option / "data").is_dir()
                    and (option / "notebooks").is_dir()
                ):
                    return option
    raise FileNotFoundError(
        "Could not locate packages/models. Start Jupyter inside the repository, or set "
        "LINGUAE_PACKAGE_ROOT=/absolute/path/to/packages/models."
    )


PACKAGE_ROOT = find_package_root()


@dataclass(frozen=True)
class Config:
    '''Single configuration object for data, model, training and export.'''

    # Data
    dataset_path: Path = PACKAGE_ROOT / "data" / "csv" / "wili-2018.csv"
    text_column: str = "text"
    label_column: str = "language"
    max_rows_per_language: int = 0  # 0 keeps every row; > 0 subsets for quick runs

    # Deterministic splitting
    train_fraction: float = 0.8
    val_fraction: float = 0.1
    test_fraction: float = 0.1

    # Encoding and model
    max_length: int = 256
    embedding_dim: int = 32
    num_filters: tuple[int, ...] = (128, 128, 128)  # one entry per kernel size
    kernel_sizes: tuple[int, ...] = (3, 5, 7)
    dropout: float = 0.3

    # Optimisation
    batch_size: int = 256
    epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    early_stopping_patience: int = 2
    num_workers: int = 0

    # Evaluation and export
    max_misclassified_examples: int = 2000
    artifact_dir: Path = PACKAGE_ROOT / "artifacts" / "linguae"
    model_version: str = "1.0.0"
    allow_overwrite: bool = False

    # Runtime
    seed: int = 42
    device: str = "auto"  # "auto", "cpu", "cuda" or "mps"


CONFIG = Config()


def seed_everything(seed: int) -> None:
    '''Seed Python, NumPy and PyTorch (CPU and CUDA) random number generators.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(CONFIG.seed)

print(f"package root  : {PACKAGE_ROOT}")
print(f"dataset       : {CONFIG.dataset_path}")
print(f"artifacts     : {CONFIG.artifact_dir}")
print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")
print(f"torch         : {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
print(f"mps available : {torch.backends.mps.is_available()}")

### 1.1 Reusable inference code (single source of truth)

The next cell defines `INFERENCE_MODULE_SOURCE`, a standalone Python module that contains the text normalisation, character encoding, `CharCNN` architecture, artifact loading and prediction helpers. The notebook executes it immediately and later writes the **same string** into `artifacts/linguae/linguae_inference.py`, so training code and exported inference code cannot drift apart.

Design decisions:

- `PAD_ID = 0` and `UNK_ID = 1` are reserved; real characters start at 2.
- Text is NFC-normalised and whitespace-collapsed, preserving case and accents.
- The vocabulary is a plain JSON-serialisable `{character: index}` mapping built only from the training split.
- The CNN uses odd kernel sizes with `padding = kernel_size // 2`, so the convolution output length matches the input length and the padding mask can be applied directly.
- Masked max pooling (`-inf` at padding positions, then max over time) stops pad characters from inflating the representation; even an all-padding row is mapped to zeros.
- `load_artifact` rebuilds the model from `config.json` and `model.pt` with `map_location`, then switches to eval mode.

In [ ]:
INFERENCE_MODULE_SOURCE = r'''"""Standalone inference utilities for the exported Linguae language detector.

The artifact directory contains this module next to ``model.pt``, ``config.json``,
``vocabulary.json`` and ``labels.json`` so that inference needs neither the
training notebook nor the training data.

Typical use::

    from linguae_inference import load_artifact, predict_text

    artifact = load_artifact("./artifacts/linguae")
    print(predict_text("Bonjour tout le monde", artifact))

The model predicts exactly one language for a whole string. It does not perform
mixed-language span detection.
"""

from __future__ import annotations

import json
import re
import unicodedata
from pathlib import Path

import torch
from torch import nn

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1
SPECIAL_TOKENS = (PAD_TOKEN, UNK_TOKEN)

_WHITESPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """NFC-normalise and collapse whitespace without changing case or accents."""
    return _WHITESPACE_RE.sub(" ", unicodedata.normalize("NFC", str(text))).strip()


def encode_text(text: str, vocabulary: dict[str, int], max_length: int) -> list[int]:
    """Encode one string as character IDs, mapping unknown characters to ``UNK_ID``."""
    characters = normalize_text(text)[:max_length]
    if not characters:
        return [UNK_ID]
    return [vocabulary.get(character, UNK_ID) for character in characters]


def pad_sequences(sequences: list[list[int]], max_length: int | None = None) -> torch.Tensor:
    """Right-pad ID sequences with ``PAD_ID`` and return a long tensor ``[batch, length]``."""
    if not sequences:
        return torch.empty((0, 0), dtype=torch.long)
    length = max(len(sequence) for sequence in sequences)
    if max_length is not None:
        length = min(length, max_length)
    length = max(length, 1)
    batch = torch.full((len(sequences), length), PAD_ID, dtype=torch.long)
    for row, sequence in enumerate(sequences):
        ids = sequence[:length]
        if ids:
            batch[row, : len(ids)] = torch.tensor(ids, dtype=torch.long)
    return batch


class CharCNN(nn.Module):
    """Character-level CNN with masked max pooling over configurable kernels."""

    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 32,
        num_filters: tuple[int, ...] = (128, 128),
        kernel_sizes: tuple[int, ...] = (3, 5, 7),
        dropout: float = 0.3,
        padding_idx: int = PAD_ID,
    ) -> None:
        super().__init__()
        if len(num_filters) != len(kernel_sizes):
            raise ValueError("num_filters and kernel_sizes must have the same length.")
        if any(kernel_size % 2 == 0 for kernel_size in kernel_sizes):
            raise ValueError("kernel_sizes must be odd so that padding preserves the length.")
        self.padding_idx = int(padding_idx)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=self.padding_idx)
        self.convolutions = nn.ModuleList(
            nn.Conv1d(embedding_dim, filters, kernel_size, padding=kernel_size // 2)
            for filters, kernel_size in zip(num_filters, kernel_sizes)
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(sum(num_filters), num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Map character IDs ``[batch, length]`` to logits ``[batch, num_classes]``."""
        mask = input_ids.ne(self.padding_idx)
        embedded = self.embedding(input_ids).transpose(1, 2)
        pooled = []
        for convolution in self.convolutions:
            features = torch.relu(convolution(embedded))
            features = features.masked_fill(~mask.unsqueeze(1), float("-inf"))
            maximum = features.max(dim=2).values
            pooled.append(torch.nan_to_num(maximum, neginf=0.0))
        representation = self.dropout(torch.cat(pooled, dim=1))
        return self.classifier(representation)


def build_model(config: dict, device: str = "cpu") -> CharCNN:
    """Construct a :class:`CharCNN` from an exported ``config.json`` mapping."""
    model = CharCNN(
        vocab_size=int(config["vocab_size"]),
        num_classes=int(config["num_classes"]),
        embedding_dim=int(config["embedding_dim"]),
        num_filters=tuple(int(value) for value in config["num_filters"]),
        kernel_sizes=tuple(int(value) for value in config["kernel_sizes"]),
        dropout=float(config["dropout"]),
        padding_idx=int(config.get("padding_idx", PAD_ID)),
    )
    return model.to(device)


def load_artifact(directory: str | Path, device: str = "cpu") -> dict:
    """Load config, vocabulary, labels and weights from an artifact directory."""
    directory = Path(directory)
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    vocabulary = json.loads((directory / "vocabulary.json").read_text(encoding="utf-8"))
    labels = json.loads((directory / "labels.json").read_text(encoding="utf-8"))
    model = build_model(config, device=device)
    state_dict = torch.load(directory / "model.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    return {
        "directory": directory,
        "config": config,
        "vocabulary": vocabulary,
        "label2idx": labels["label2idx"],
        "idx2label": {int(index): label for index, label in labels["idx2label"].items()},
        "model": model,
        "device": torch.device(device),
    }


def predict_text(text: str, artifact: dict, top_k: int = 3) -> dict:
    """Predict one language label for a whole string and return top-k scores."""
    model = artifact["model"]
    max_length = int(artifact["config"]["max_length"])
    encoded = encode_text(text, artifact["vocabulary"], max_length)
    batch = pad_sequences([encoded], max_length=max_length).to(artifact["device"])
    with torch.inference_mode():
        probabilities = torch.softmax(model(batch), dim=1)[0]
    k = max(1, min(int(top_k), probabilities.numel()))
    scores, indices = torch.topk(probabilities, k)
    predictions = [
        {"language": artifact["idx2label"][int(index)], "score": float(score)}
        for score, index in zip(scores.tolist(), indices.tolist())
    ]
    return {
        "text": text,
        "language": predictions[0]["language"],
        "confidence": predictions[0]["score"],
        "top_k": predictions,
    }


def predict_batch(texts: list[str], artifact: dict, batch_size: int = 256) -> list[dict]:
    """Predict one language label per string in a list."""
    model = artifact["model"]
    max_length = int(artifact["config"]["max_length"])
    predictions = []
    texts = list(texts)
    for start in range(0, len(texts), batch_size):
        chunk = texts[start : start + batch_size]
        sequences = [encode_text(text, artifact["vocabulary"], max_length) for text in chunk]
        batch = pad_sequences(sequences, max_length=max_length).to(artifact["device"])
        with torch.inference_mode():
            probabilities = torch.softmax(model(batch), dim=1)
        best = probabilities.argmax(dim=1)
        confidences = probabilities.gather(1, best.unsqueeze(1)).squeeze(1)
        predictions.extend(
            {
                "text": text,
                "language": artifact["idx2label"][int(index)],
                "confidence": float(confidence),
            }
            for text, index, confidence in zip(chunk, best.tolist(), confidences.tolist())
        )
    return predictions
'''

_module_namespace: dict[str, object] = {}
exec(compile(INFERENCE_MODULE_SOURCE, "linguae_inference.py", "exec"), _module_namespace)

CharCNN = _module_namespace["CharCNN"]
build_model = _module_namespace["build_model"]
load_artifact = _module_namespace["load_artifact"]
predict_text = _module_namespace["predict_text"]
predict_batch = _module_namespace["predict_batch"]
normalize_text = _module_namespace["normalize_text"]
encode_text = _module_namespace["encode_text"]
pad_sequences = _module_namespace["pad_sequences"]
PAD_ID = _module_namespace["PAD_ID"]
UNK_ID = _module_namespace["UNK_ID"]
SPECIAL_TOKENS = _module_namespace["SPECIAL_TOKENS"]

print(f"inference module lines: {len(INFERENCE_MODULE_SOURCE.splitlines())}")
print(f"special tokens: {SPECIAL_TOKENS} (pad={PAD_ID}, unk={UNK_ID})")

## 2. Load and validate the dataset

`CONFIG.dataset_path` is resolved relative to the `packages/models` directory, not the current working directory, so the notebook behaves the same from the repository root or from `notebooks/`.

Validation rules:

- The file must exist and the configured text/label columns must be present.
- Values are read as strings with `keep_default_na=False`, so text such as `"NA"` is not silently converted into a missing value.
- Rows are rejected only for explicit reasons: empty text after stripping, empty language label, or empty text after normalisation. Rejected rows are collected in `invalid_df` with a `reason` column and reported, never dropped silently.

In [ ]:
def load_dataset(path: Path, text_column: str, label_column: str) -> pd.DataFrame:
    '''Read the CSV as strings, validate the schema and return text/language columns.'''
    if not path.is_file():
        raise FileNotFoundError(
            f"Dataset not found at {path}. Build it with the WiLI text-to-CSV notebook "
            "or point CONFIG.dataset_path at data/processed/dataset.csv."
        )
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[])
    missing = [column for column in (text_column, label_column) if column not in frame.columns]
    if missing:
        raise ValueError(
            f"Missing required column(s) {missing}; found columns {list(frame.columns)}. "
            "Expected 'text' and 'language' (configurable via CONFIG)."
        )
    return frame[[text_column, label_column]].rename(
        columns={text_column: "text", label_column: "language"}
    )


raw_df = load_dataset(CONFIG.dataset_path, CONFIG.text_column, CONFIG.label_column)
print(f"rows: {len(raw_df):,}  columns: {list(raw_df.columns)}")
print(f"missing values per column: {raw_df.isna().sum().to_dict()}")
raw_df.head(3)

### 2.1 Normalisation

Normalisation is deliberately conservative:

- `unicodedata.normalize("NFC", ...)` gives composed characters a canonical form so the same visible character is not split into different code points.
- Runs of whitespace become a single space and leading/trailing whitespace is removed.
- Case is preserved and accents are kept. Lowercasing or stripping diacritics destroys language-specific signal (for example `ç` vs `c`, or case-rich languages such as German nouns).
- The raw text and raw label are retained in `raw_text` / `raw_language` for auditing.

If `CONFIG.max_rows_per_language > 0`, the cleaned data is deterministically subsampled per language for quick experiments; otherwise every valid row is used.

In [ ]:
_WHITESPACE_RE = re.compile(r"\s+")


def normalise_text(value: object) -> str:
    '''NFC-normalise and collapse whitespace; case and accents are preserved.'''
    return _WHITESPACE_RE.sub(" ", unicodedata.normalize("NFC", str(value))).strip()


def validate_and_normalise(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    '''Return (clean, invalid, report) without silently discarding invalid records.'''
    text = frame["text"].astype(str)
    label = frame["language"].astype(str)

    normalised = pd.DataFrame(
        {
            "text": text.map(normalise_text),
            "language": label.str.strip(),
            "raw_text": text,
            "raw_language": label,
        }
    )

    empty_text = text.str.strip().eq("")
    empty_label = label.str.strip().eq("")
    empty_after_normalisation = normalised["text"].eq("")
    invalid_mask = empty_text | empty_label | empty_after_normalisation

    invalid = frame.loc[invalid_mask].copy()
    if len(invalid):
        invalid["reason"] = np.select(
            [
                empty_text[invalid_mask].to_numpy(),
                empty_label[invalid_mask].to_numpy(),
                empty_after_normalisation[invalid_mask].to_numpy(),
            ],
            ["empty text after strip", "empty language label", "empty text after normalisation"],
            default="unknown",
        )
    else:
        invalid["reason"] = pd.Series(dtype="object")

    clean = normalised.loc[~invalid_mask].copy()
    label_counts = clean["language"].value_counts()

    report = {
        "input_rows": int(len(frame)),
        "valid_rows": int(len(clean)),
        "invalid_rows": int(invalid_mask.sum()),
        "missing_values": {column: int(frame[column].isna().sum()) for column in frame.columns},
        "empty_text_rows": int(empty_text.sum()),
        "empty_label_rows": int(empty_label.sum()),
        "empty_after_normalisation_rows": int(empty_after_normalisation.sum()),
        "rows_changed_by_normalisation": int((normalised["text"] != text).sum()),
        "unique_languages": int(clean["language"].nunique()),
        "duplicate_text_rows": int(clean.duplicated(subset=["text"]).sum()),
        "duplicate_text_label_rows": int(clean.duplicated(subset=["text", "language"]).sum()),
        "texts_with_conflicting_labels": int(
            clean.groupby("text")["language"].nunique().gt(1).sum()
        ),
        "min_examples_per_language": int(label_counts.min()) if len(label_counts) else 0,
        "max_examples_per_language": int(label_counts.max()) if len(label_counts) else 0,
    }

    if clean.empty:
        raise ValueError("No valid rows remain after validation; check the dataset.")
    if report["empty_label_rows"]:
        raise ValueError(
            f"{report['empty_label_rows']} row(s) have an empty language label; "
            "inspect invalid_df and fix the source data."
        )
    return clean, invalid, report


clean_df, invalid_df, validation_report = validate_and_normalise(raw_df)
print(json.dumps(validation_report, indent=2))
if not invalid_df.empty:
    print(f"\nInvalid rows kept for inspection ({len(invalid_df):,}):")
    display(invalid_df.head(10))

if CONFIG.max_rows_per_language > 0:
    rng = np.random.default_rng(CONFIG.seed)
    selected = []
    for _, language_group in clean_df.groupby("language", sort=True):
        if len(language_group) > CONFIG.max_rows_per_language:
            keep = np.sort(
                rng.choice(len(language_group), size=CONFIG.max_rows_per_language, replace=False)
            )
            selected.append(language_group.iloc[keep])
        else:
            selected.append(language_group)
    clean_df = pd.concat(selected).sort_index()
    print(
        f"\nsubsampled to at most {CONFIG.max_rows_per_language:,} rows per language: "
        f"{len(clean_df):,} rows"
    )
else:
    print(f"\nusing all {len(clean_df):,} valid rows")


### 2.2 Dataset quality report

The report below covers the checks required before preprocessing:

- number of unique labels and examples per language (class imbalance),
- duplicate rows and duplicate texts with conflicting labels,
- missing/invalid values,
- character-length distribution, including extreme outliers such as very long Wikipedia articles,
- a sorted view of per-language counts to expose under-represented classes.

WiLI-2018 is balanced by construction (1,000 paragraphs per language), but the code does not assume that.

In [ ]:
label_counts = clean_df["language"].value_counts().sort_index()
text_character_lengths = clean_df["text"].str.len()

print(f"unique labels: {clean_df['language'].nunique():,}")
print(
    "examples per label: "
    f"min={label_counts.min():,} median={int(label_counts.median()):,} max={label_counts.max():,}"
)
print(
    "character length: "
    f"min={text_character_lengths.min():,} median={text_character_lengths.median():.0f} "
    f"p90={text_character_lengths.quantile(0.9):.0f} p99={text_character_lengths.quantile(0.99):.0f} "
    f"max={text_character_lengths.max():,}"
)
print(f"duplicate text rows: {clean_df.duplicated(subset=['text']).sum():,}")
print(f"conflicting duplicate texts: {validation_report['texts_with_conflicting_labels']:,}")

print("\nmost common labels:")
print(label_counts.sort_values(ascending=False).head(10).to_string())
print("\nrarest labels:")
print(label_counts.sort_values().head(10).to_string())

figure, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(text_character_lengths.clip(upper=1500), bins=60, color="steelblue")
axes[0].set(
    title="Character length distribution (clipped at 1500)",
    xlabel="characters",
    ylabel="examples",
)
axes[1].bar(
    range(len(label_counts)),
    label_counts.sort_values(ascending=False).to_numpy(),
    color="darkorange",
)
axes[1].set(title="Examples per language (sorted)", xlabel="language rank", ylabel="examples")
plt.tight_layout()
plt.show()

text_character_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("characters")

### 2.3 Quality summary and next steps

**What to check before modelling**

- `invalid_rows` should be zero or explained; inspect `invalid_df` when it is not.
- `texts_with_conflicting_labels` should be zero. Conflicting duplicates are kept (they stay in the same split) but they limit achievable accuracy.
- The per-language counts and the sorted bar chart reveal class imbalance. Accuracy alone is misleading when classes are imbalanced, so macro-F1 is reported later.
- Very long documents are truncated during encoding. The truncation rate is printed in section 5; language identification usually survives truncation because scripts and frequent characters repeat.

**Next steps**

1. Split per language with duplicate texts grouped together (section 3).
2. Build the character vocabulary from the training split only (section 4).
3. Encode, pad and load batches (sections 5–6).
4. Train the character CNN with early stopping (sections 7–8).
5. Evaluate once on the test split, then export the artifact (sections 9–10).

## 3. Train/validation/test split

The split is deterministic and grouped:

- Each language is split separately using seeded NumPy permutations, so every language appears in all three splits and evaluation labels always exist in training.
- Exact duplicate texts are grouped first (`group_id`), so the same paragraph can never appear in both training and evaluation. This prevents leakage from duplicated Wikipedia content.
- Splits use 80% / 10% / 10% by default (`CONFIG.train_fraction`, `val_fraction`, `test_fraction`).
- A language with fewer than three unique texts cannot be split and raises a helpful error.
- The code asserts that evaluation languages are a subset of training languages and that no text overlaps between splits.

The test split is not used for vocabulary building, hyperparameter selection, early stopping or checkpoint selection.

In [ ]:
def split_dataset(frame: pd.DataFrame, config: Config) -> tuple[pd.DataFrame, dict]:
    '''Deterministically split per language while keeping duplicate texts together.'''
    fractions = (config.train_fraction, config.val_fraction, config.test_fraction)
    if any(fraction <= 0 for fraction in fractions) or abs(sum(fractions) - 1.0) > 1e-6:
        raise ValueError(
            f"train/val/test fractions must be positive and sum to 1.0, got {fractions}"
        )

    frame = frame.copy()
    frame["group_id"] = frame.groupby("text", sort=False).ngroup()
    label_of_group = frame.drop_duplicates("group_id").set_index("group_id")["language"]

    rng = np.random.default_rng(config.seed)
    split_of_group = pd.Series(index=label_of_group.index, dtype="object")

    for language, group_ids in label_of_group.groupby(label_of_group).groups.items():
        shuffled = rng.permutation(np.asarray(group_ids))
        count = len(shuffled)
        n_val = max(1, int(round(count * config.val_fraction)))
        n_test = max(1, int(round(count * config.test_fraction)))
        n_train = count - n_val - n_test
        if n_train < 1:
            raise ValueError(
                f"Language {language!r} has only {count} unique text(s); at least 3 are "
                "needed for a train/validation/test split. Increase the dataset or adjust "
                "the split fractions."
            )
        split_of_group.loc[shuffled[:n_train]] = "train"
        split_of_group.loc[shuffled[n_train : n_train + n_val]] = "validation"
        split_of_group.loc[shuffled[n_train + n_val :]] = "test"

    frame["split"] = frame["group_id"].map(split_of_group)
    if frame["split"].isna().any():
        raise RuntimeError("Some examples were not assigned to a split.")

    train_labels = set(frame.loc[frame["split"] == "train", "language"])
    evaluation_labels = set(frame.loc[frame["split"].isin(["validation", "test"]), "language"])
    missing_from_train = evaluation_labels - train_labels
    if missing_from_train:
        raise ValueError(
            f"Evaluation languages missing from training: {sorted(missing_from_train)[:10]}"
        )

    overlaps = {}
    for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
        left_texts = set(frame.loc[frame["split"] == left, "text"])
        right_texts = set(frame.loc[frame["split"] == right, "text"])
        overlaps[f"{left}|{right}"] = len(left_texts & right_texts)
    if any(overlaps.values()):
        raise RuntimeError(f"Duplicate texts leaked across splits: {overlaps}")

    per_label = frame.pivot_table(
        index="language", columns="split", values="group_id", aggfunc="nunique", fill_value=0
    )
    details = {
        "rows_per_split": {
            split: int((frame["split"] == split).sum())
            for split in ("train", "validation", "test")
        },
        "languages_per_split": {
            split: int(frame.loc[frame["split"] == split, "language"].nunique())
            for split in ("train", "validation", "test")
        },
        "text_overlap": overlaps,
        "examples_per_language_by_split": (
            per_label.describe().loc[["min", "max"]].round(1).to_dict()
        ),
    }
    return frame, details


clean_df, split_details = split_dataset(clean_df, CONFIG)
print(json.dumps(split_details, indent=2))
print("\nrows per split:")
print(clean_df.groupby("split").size().to_string())


## 4. Character vocabulary (training split only)

- The vocabulary is the sorted set of characters observed in the **training split**, plus `<pad>` (0) and `<unk>` (1).
- Validation and test characters that never appeared in training are mapped to `<unk>`, which lets us measure how much evaluation text the vocabulary misses.
- Sorting gives a stable, reproducible index assignment that does not depend on pandas row order.
- The label mapping is sorted alphabetically for the same reason, so label index `i` always refers to `labels[i]`.
- All Unicode scripts and accents are kept as individual characters: no transliteration, no lowercasing, no `unicodedata` category filtering.

The `idx2label` / `label2idx` mappings are written to `data/processed/labels.json` and are exported again inside the artifact.

In [ ]:
def build_vocabulary(texts: list[str]) -> dict[str, int]:
    '''Build a character vocabulary; indices 0 and 1 are reserved for pad and unk.'''
    characters = sorted({character for text in texts for character in text})
    vocabulary = {token: index for index, token in enumerate(SPECIAL_TOKENS)}
    for character in characters:
        vocabulary[character] = len(vocabulary)
    return vocabulary


train_df = clean_df.loc[clean_df["split"] == "train"].reset_index(drop=True)
validation_df = clean_df.loc[clean_df["split"] == "validation"].reset_index(drop=True)
test_df = clean_df.loc[clean_df["split"] == "test"].reset_index(drop=True)

vocabulary = build_vocabulary(train_df["text"].tolist())

unknown_character_rate = {}
for split_name, split_frame in (("train", train_df), ("validation", validation_df), ("test", test_df)):
    total_characters = int(split_frame["text"].str.len().sum())
    unseen_characters = sum(
        character not in vocabulary
        for text in split_frame["text"]
        for character in text
    )
    unknown_character_rate[split_name] = (
        round(unseen_characters / total_characters, 6) if total_characters else 0.0
    )

print(f"vocabulary size: {len(vocabulary):,} ids ({len(vocabulary) - len(SPECIAL_TOKENS):,} characters)")
print(f"unknown character rate: {unknown_character_rate}")
print(f"first characters: {[character for character in vocabulary if character not in SPECIAL_TOKENS][:40]}")


## 5. Encode strings as character-ID sequences

Every string becomes a fixed-width `int32` matrix row of length `CONFIG.max_length`:

1. Normalise once with the same function used at inference time.
2. Map each character to its vocabulary id, unknown characters to `UNK_ID`.
3. Truncate after `max_length` characters (WiLI paragraphs are long; the beginning of a paragraph is usually sufficient for language identification).
4. Right-pad shorter strings with `PAD_ID = 0`. Padding is masked out of the convolution pooling, so it never contributes to the representation.

Character length (before truncation) is kept alongside each row for the length-based error analysis. The truncation rate printed below shows how many rows hit the limit.

In [ ]:
def encode_frame(
    frame: pd.DataFrame, vocabulary: dict[str, int], max_length: int
) -> tuple[np.ndarray, np.ndarray]:
    '''Encode texts as right-padded int32 ID matrices and return raw character lengths.'''
    ids = np.full((len(frame), max_length), PAD_ID, dtype=np.int32)
    lengths = np.zeros(len(frame), dtype=np.int32)
    for row, text in enumerate(frame["text"]):
        encoded = encode_text(text, vocabulary, max_length)
        ids[row, : len(encoded)] = encoded
        lengths[row] = len(text)
    return ids, lengths


encoded: dict[str, np.ndarray] = {}
character_lengths: dict[str, np.ndarray] = {}
for split_name, split_frame in (("train", train_df), ("validation", validation_df), ("test", test_df)):
    encoded[split_name], character_lengths[split_name] = encode_frame(
        split_frame, vocabulary, CONFIG.max_length
    )
    truncated = int((character_lengths[split_name] > CONFIG.max_length).sum())
    print(
        f"{split_name:<10} shape={encoded[split_name].shape} "
        f"chars median={np.median(character_lengths[split_name]):.0f} "
        f"p95={np.percentile(character_lengths[split_name], 95):.0f} "
        f"truncated={truncated:,} ({truncated / len(split_frame):.1%})"
    )


## 6. PyTorch datasets and data loaders

- `LanguageDataset` stores the padded ID matrix (`int32`, converted to `int64` per item) and integer labels, so the full dataset is not duplicated as `int64` tensors in memory.
- The training loader shuffles with a seeded generator; validation and test loaders are unshuffled so prediction order matches `validation_df` / `test_df` row order.
- `num_workers=0` keeps execution identical across platforms (no worker RNG divergence); increase `CONFIG.num_workers` on Linux if I/O becomes a bottleneck.
- Raw text and labels remain in the DataFrames for evaluation and error analysis.

In [ ]:
class LanguageDataset(Dataset):
    '''Character-ID matrix and integer language label per example.'''

    def __init__(self, ids: np.ndarray, labels: np.ndarray) -> None:
        if len(ids) != len(labels):
            raise ValueError(f"ids and labels length mismatch: {len(ids)} != {len(labels)}")
        self.ids = ids
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return (
            torch.from_numpy(self.ids[index].astype(np.int64)),
            torch.tensor(int(self.labels[index]), dtype=torch.long),
        )


labels = sorted(clean_df["language"].unique())
label2idx = {label: index for index, label in enumerate(labels)}
idx2label = {index: label for label, index in label2idx.items()}

datasets: dict[str, LanguageDataset] = {}
loaders: dict[str, DataLoader] = {}
for split_name, split_frame in (("train", train_df), ("validation", validation_df), ("test", test_df)):
    label_ids = split_frame["language"].map(label2idx).to_numpy(dtype=np.int64)
    datasets[split_name] = LanguageDataset(encoded[split_name], label_ids)
    loaders[split_name] = DataLoader(
        datasets[split_name],
        batch_size=CONFIG.batch_size,
        shuffle=split_name == "train",
        num_workers=CONFIG.num_workers,
        pin_memory=torch.cuda.is_available(),
        generator=torch.Generator().manual_seed(CONFIG.seed) if split_name == "train" else None,
    )

labels_path = PACKAGE_ROOT / "data" / "processed" / "labels.json"
labels_path.parent.mkdir(parents=True, exist_ok=True)
labels_path.write_text(
    json.dumps(
        {
            "label2idx": label2idx,
            "idx2label": {str(index): label for index, label in idx2label.items()},
        },
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print(f"languages: {len(labels):,}")
print("batches: " + ", ".join(f"{name}={len(loader):,}" for name, loader in loaders.items()))
print(f"label mappings written to {labels_path}")
preview_inputs, preview_targets = next(iter(loaders["train"]))
print(f"batch input shape {tuple(preview_inputs.shape)} dtype {preview_inputs.dtype}")
print(
    f"batch target shape {tuple(preview_targets.shape)} "
    f"example labels {[idx2label[int(index)] for index in preview_targets[:5]]}"
)


## 7. Character-level CNN architecture

The model is built from scratch with PyTorch:

| Stage | Operation | Output shape |
| --- | --- | --- |
| Input | character IDs | `[batch, length]` |
| Embedding | `nn.Embedding(vocab_size, embedding_dim)`, padding index 0 | `[batch, length, embedding_dim]` |
| Transpose | channels-first for 1D convolution | `[batch, embedding_dim, length]` |
| Convolution block × k | `Conv1d(embedding_dim, filters, kernel)` → ReLU | `[batch, filters, length]` |
| Masked max pooling | padding positions set to `-inf`, max over time | `[batch, filters]` per kernel |
| Concatenate | all kernel branches | `[batch, sum(filters)]` |
| Dropout | regularisation | `[batch, sum(filters)]` |
| Classifier | `nn.Linear(sum(filters), num_classes)` | `[batch, num_classes]` |

Notes:

- Kernel sizes are odd and `padding = kernel_size // 2`, so the time dimension is preserved and the padding mask aligns with convolution outputs.
- Kernels of 3, 5 and 7 characters capture short script cues (diacritics, digraphs) and slightly longer patterns simultaneously.
- Masked max pooling uses `-inf` for `PAD_ID` positions, so trailing padding cannot dominate. The `-inf` values are converted to zero for safety.
- The model returns **logits**. `CrossEntropyLoss` applies log-softmax internally; softmax is applied only for confidence reporting after training.
- No BatchNorm is used, which keeps the model stable with small evaluation batches and simple to export.
- Device selection prefers CUDA, then Apple Silicon MPS, then CPU. `CONFIG.device` can force any of them; forcing an unavailable device raises an explicit error.

In [ ]:
def select_device(preference: str) -> torch.device:
    '''Pick CUDA, Apple MPS or CPU according to availability and configuration.'''
    requested = preference.lower()
    if requested not in {"auto", "cpu", "cuda", "mps"}:
        raise ValueError(f"Unknown device preference {preference!r}")
    if requested in {"auto", "cuda"} and torch.cuda.is_available():
        return torch.device("cuda")
    if requested in {"auto", "mps"} and torch.backends.mps.is_available():
        return torch.device("mps")
    if requested == "cuda":
        raise RuntimeError("CUDA was requested but is not available.")
    if requested == "mps":
        raise RuntimeError("MPS was requested but is not available.")
    return torch.device("cpu")


DEVICE = select_device(CONFIG.device)

model = CharCNN(
    vocab_size=len(vocabulary),
    num_classes=len(labels),
    embedding_dim=CONFIG.embedding_dim,
    num_filters=CONFIG.num_filters,
    kernel_sizes=CONFIG.kernel_sizes,
    dropout=CONFIG.dropout,
    padding_idx=PAD_ID,
).to(DEVICE)

example_batch = pad_sequences([[2, 3, 4, 5, 6, 7], [2, 3]], max_length=CONFIG.max_length).to(DEVICE)
with torch.no_grad():
    example_embeddings = model.embedding(example_batch)
    example_logits = model(example_batch)

print(model)
print(f"\ndevice            : {DEVICE}")
print(f"input ids         : {tuple(example_batch.shape)}  (batch, length)")
print(f"embeddings        : {tuple(example_embeddings.shape)}  (batch, length, embedding_dim)")
print(f"logits            : {tuple(example_logits.shape)}  (batch, num_classes)")
print(f"trainable params  : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 8. Training loop

- **Loss:** `nn.CrossEntropyLoss` on raw logits (no softmax before the loss).
- **Optimiser:** AdamW with configurable learning rate and weight decay, plus gradient-norm clipping.
- **Epoch:** one pass over the shuffled training loader in `model.train()` mode, then validation in `model.eval()` under `torch.no_grad()`.
- **Metrics:** mean training loss, mean validation loss and validation accuracy per epoch.
- **Early stopping:** validation loss is the selection signal. The best `state_dict` is copied to CPU memory; training stops after `early_stopping_patience` epochs without improvement, and the best checkpoint is restored afterwards.
- **Safety checks:** non-finite losses raise immediately, and empty loaders raise instead of silently training on nothing.
- **No test leakage:** the test split is untouched until section 9; early stopping never sees it.

Reproducibility caveats: seeds make the split, shuffling and initialisation repeatable, but CUDA/MPS kernels, mixed precision (not used here) and library versions can still cause tiny numerical differences. CPU training is the most deterministic option.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Run one training epoch and return mean loss and accuracy.'''
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for input_ids, targets in loader:
        if input_ids.numel() == 0:
            continue
        input_ids = input_ids.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids)
        loss = criterion(logits, targets)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss: {loss.item()}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG.grad_clip_norm)
        optimizer.step()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        correct += int((logits.argmax(dim=1) == targets).sum().item())
        total += batch_size

    if total == 0:
        raise RuntimeError("Training loader produced no examples.")
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Compute mean loss and accuracy in eval mode without gradients.'''
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for input_ids, targets in loader:
        if input_ids.numel() == 0:
            continue
        input_ids = input_ids.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(input_ids)
        loss = criterion(logits, targets)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite validation loss: {loss.item()}")

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        correct += int((logits.argmax(dim=1) == targets).sum().item())
        total += batch_size

    if total == 0:
        raise RuntimeError("Evaluation loader produced no examples.")
    return total_loss / total, correct / total


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay
)

history: list[dict] = []
best_state: dict[str, torch.Tensor] | None = None
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(1, CONFIG.epochs + 1):
    started = time.time()
    train_loss, train_accuracy = train_one_epoch(
        model, loaders["train"], optimizer, criterion, DEVICE
    )
    val_loss, val_accuracy = evaluate_loader(model, loaders["validation"], criterion, DEVICE)
    elapsed = time.time() - started

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {
            name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()
        }

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "seconds": round(elapsed, 1),
            "improved": improved,
        }
    )
    print(
        f"epoch {epoch:>2}/{CONFIG.epochs} | "
        f"train loss {train_loss:.4f} acc {train_accuracy:.4f} | "
        f"val loss {val_loss:.4f} acc {val_accuracy:.4f} | "
        f"{elapsed:5.1f}s {'*' if improved else ''}"
    )

    if epoch - best_epoch >= CONFIG.early_stopping_patience:
        print(
            f"early stopping at epoch {epoch}: validation loss has not improved for "
            f"{CONFIG.early_stopping_patience} epoch(s)."
        )
        break

if best_state is None:
    raise RuntimeError("Training finished without a checkpoint.")

model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
best_row = history_df.loc[history_df["epoch"] == best_epoch].iloc[0]
print(
    f"\nbest epoch {best_epoch}: val loss {best_row['val_loss']:.4f}, "
    f"val accuracy {best_row['val_accuracy']:.4f}; restored best checkpoint."
)
history_df


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
axes[0].axvline(best_epoch, color="grey", linestyle="--", label=f"best epoch ({best_epoch})")
axes[0].set(title="Cross-entropy loss", xlabel="epoch", ylabel="loss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="train")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="validation")
axes[1].axvline(best_epoch, color="grey", linestyle="--")
axes[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

last_row = history_df.iloc[-1]
print(
    f"epochs run: {len(history_df)} | best epoch: {best_epoch} | "
    f"final train loss/accuracy: {last_row['train_loss']:.4f}/{last_row['train_accuracy']:.4f} | "
    f"final validation loss/accuracy: {last_row['val_loss']:.4f}/{last_row['val_accuracy']:.4f}"
)


## 9. Final evaluation on the held-out test set

The best validation checkpoint is evaluated **once** on the test split. Nothing below feeds back into training, vocabulary building or model selection.

- Accuracy, per-language precision/recall/F1 and macro-F1 are computed with NumPy and plain Python (no scikit-learn).
- The confusion matrix uses language codes on both axes and a log scale, because off-diagonal counts are small.
- Misclassified examples are shown with expected label, predicted label and confidence.
- Accuracy is analysed by input length and by language, so weak classes are visible.
- Confidence summaries show how softmax scores relate to correctness.
- Leakage, duplicate and imbalance checks run after evaluation; unrealistic results are flagged.

**Caveat:** softmax scores are ranking scores from this model, not calibrated probabilities. A score of 0.9 does not mean the model is correct 90% of the time.

In [ ]:
def prepare_artifact_dir(config: Config) -> Path:
    '''Create the artifact directory unless an existing version must not be overwritten.'''
    directory = config.artifact_dir
    if directory.exists() and any(directory.iterdir()) and not config.allow_overwrite:
        raise RuntimeError(
            f"{directory} already contains files. Bump CONFIG.model_version, choose another "
            "CONFIG.artifact_dir, or set CONFIG.allow_overwrite = True to replace it."
        )
    directory.mkdir(parents=True, exist_ok=True)
    return directory


ARTIFACT_DIR = prepare_artifact_dir(CONFIG)

model.eval()
test_probability_batches = []
with torch.inference_mode():
    for input_ids, _ in loaders["test"]:
        if input_ids.numel() == 0:
            continue
        logits = model(input_ids.to(DEVICE))
        test_probability_batches.append(torch.softmax(logits, dim=1).cpu())

if not test_probability_batches:
    raise RuntimeError("Test loader produced no examples.")
test_probabilities = torch.cat(test_probability_batches, dim=0)

y_true = test_df["language"].map(label2idx).to_numpy(dtype=np.int64)
if len(test_probabilities) != len(y_true):
    raise RuntimeError(
        f"Prediction/truth length mismatch: {len(test_probabilities)} != {len(y_true)}"
    )
y_pred = test_probabilities.argmax(dim=1).numpy()
y_confidence = test_probabilities.max(dim=1).values.numpy()

print(f"test examples : {len(y_true):,}")
print(f"test accuracy : {(y_pred == y_true).mean():.4f}")
print(f"artifact dir  : {ARTIFACT_DIR}")


In [ ]:
def classification_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, label_names: list[str]
) -> tuple[dict, pd.DataFrame, np.ndarray]:
    '''Accuracy, per-class precision/recall/F1 and macro-F1 using NumPy only.'''
    matrix = np.zeros((len(label_names), len(label_names)), dtype=np.int64)
    np.add.at(matrix, (y_true, y_pred), 1)

    true_positive = np.diag(matrix)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)

    precision = np.divide(
        true_positive, predicted, out=np.zeros(len(label_names), dtype=float), where=predicted > 0
    )
    recall = np.divide(
        true_positive, support, out=np.zeros(len(label_names), dtype=float), where=support > 0
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(len(label_names), dtype=float),
        where=(precision + recall) > 0,
    )

    present = support > 0
    per_language = pd.DataFrame(
        {
            "language": label_names,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )
    summary = {
        "accuracy": float(true_positive.sum() / matrix.sum()),
        "macro_precision": float(precision[present].mean()),
        "macro_recall": float(recall[present].mean()),
        "macro_f1": float(f1[present].mean()),
        "weighted_f1": float((f1[present] * support[present]).sum() / support[present].sum()),
        "examples": int(matrix.sum()),
        "languages": int(present.sum()),
    }
    return summary, per_language, matrix


metric_summary, per_language_metrics, confusion = classification_metrics(
    y_true, y_pred, labels
)
print(json.dumps(metric_summary, indent=2))

print("\nworst 10 languages by F1 (support shown):")
display(per_language_metrics.nsmallest(10, "f1").round(4))
print("best 10 languages by F1:")
display(per_language_metrics.nlargest(10, "f1").round(4))


In [ ]:
figure, axis = plt.subplots(figsize=(26, 24))
image = axis.imshow(confusion, cmap="Blues", norm=LogNorm(vmin=1, vmax=max(int(confusion.max()), 1)))
axis.set_xticks(range(len(labels)))
axis.set_yticks(range(len(labels)))
axis.set_xticklabels(labels, rotation=90, fontsize=4)
axis.set_yticklabels(labels, fontsize=4)
axis.set(
    title="Confusion matrix — test set (row = expected, column = predicted)",
    xlabel="predicted language",
    ylabel="expected language",
)
figure.colorbar(image, ax=axis, fraction=0.025, pad=0.01)
plt.tight_layout()
plt.show()

off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
confusions = []
for flat_index in np.argsort(off_diagonal, axis=None)[::-1][:10]:
    expected_index, predicted_index = np.unravel_index(flat_index, off_diagonal.shape)
    count = int(off_diagonal[expected_index, predicted_index])
    if count == 0:
        break
    confusions.append(
        {
            "expected": labels[expected_index],
            "predicted": labels[predicted_index],
            "count": count,
        }
    )
print("most frequent confusions:")
display(pd.DataFrame(confusions))


In [ ]:
results = pd.DataFrame(
    {
        "text": test_df["text"],
        "expected": test_df["language"],
        "predicted": [labels[index] for index in y_pred],
        "confidence": y_confidence,
        "expected_idx": y_true,
        "predicted_idx": y_pred,
        "char_length": test_df["text"].str.len(),
    }
)
results["correct"] = results["expected"] == results["predicted"]

misclassified = (
    results.loc[~results["correct"]]
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)
print(
    f"misclassified: {len(misclassified):,} / {len(results):,} "
    f"({len(misclassified) / len(results):.2%})"
)
display(misclassified.assign(text=lambda frame: frame["text"].str.slice(0, 110)).head(15))

confidence_summary = {
    "mean_confidence": float(results["confidence"].mean()),
    "median_confidence": float(results["confidence"].median()),
    "share_confidence_at_least_0_9": float((results["confidence"] >= 0.9).mean()),
    "accuracy_when_confidence_at_least_0_9": (
        float(results.loc[results["confidence"] >= 0.9, "correct"].mean())
        if (results["confidence"] >= 0.9).any()
        else None
    ),
    "accuracy_when_confidence_below_0_5": (
        float(results.loc[results["confidence"] < 0.5, "correct"].mean())
        if (results["confidence"] < 0.5).any()
        else None
    ),
    "mean_confidence_correct": float(results.loc[results["correct"], "confidence"].mean()),
    "mean_confidence_incorrect": (
        float(results.loc[~results["correct"], "confidence"].mean())
        if (~results["correct"]).any()
        else None
    ),
}
print(json.dumps(confidence_summary, indent=2))
print("Softmax scores are not calibrated probabilities; treat them as ranking scores.")


### 9.1 Analysis by input length and data-quality checks

- Accuracy by character-length bucket shows whether short strings (tweets, titles) are harder than long paragraphs.
- The point-biserial correlation between character length and correctness gives a single summary of the trend.
- Leakage checks confirm that no exact text appears in two splits and that no evaluation language is absent from training.
- Class-imbalance and near-perfect-accuracy checks flag results that deserve suspicion rather than celebration.

In [ ]:
length_bins = [0, 100, 200, 400, 800, 1600, np.inf]
length_bucket_labels = ["<100", "100-199", "200-399", "400-799", "800-1599", ">=1600"]
results["length_bucket"] = pd.cut(
    results["char_length"], bins=length_bins, labels=length_bucket_labels, right=False
)

length_summary = (
    results.groupby("length_bucket", observed=True)
    .agg(examples=("correct", "size"), accuracy=("correct", "mean"))
    .reset_index()
)
length_summary["share"] = length_summary["examples"] / len(results)
print(length_summary.round(4).to_string(index=False))

length_correlation = float(
    np.corrcoef(results["char_length"].to_numpy(), results["correct"].to_numpy().astype(float))[0, 1]
)
print(f"\ncorrelation between character length and correctness: {length_correlation:.3f}")

figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(
    length_summary["length_bucket"].astype(str),
    length_summary["accuracy"],
    color="seagreen",
)
axis.set(title="Test accuracy by input length", xlabel="characters", ylabel="accuracy", ylim=(0, 1.02))
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

test_count_min = int(test_df["language"].value_counts().min())
test_count_max = int(test_df["language"].value_counts().max())
leakage_checks = {
    "duplicate_texts_train_vs_validation": len(set(train_df["text"]) & set(validation_df["text"])),
    "duplicate_texts_train_vs_test": len(set(train_df["text"]) & set(test_df["text"])),
    "duplicate_texts_validation_vs_test": len(set(validation_df["text"]) & set(test_df["text"])),
    "duplicate_rows_test": int(test_df.duplicated(subset=["text", "language"]).sum()),
    "evaluation_languages_missing_from_train": len(
        (set(validation_df["language"]) | set(test_df["language"]))
        - set(train_df["language"])
    ),
    "test_examples_per_language_min": test_count_min,
    "test_examples_per_language_max": test_count_max,
    "class_imbalance_ratio": round(test_count_max / test_count_min, 3) if test_count_min else None,
    "suspiciously_high_accuracy": bool(metric_summary["accuracy"] > 0.999),
    "test_examples": int(len(test_df)),
}
print(json.dumps(leakage_checks, indent=2))

leaked = max(
    leakage_checks["duplicate_texts_train_vs_validation"],
    leakage_checks["duplicate_texts_train_vs_test"],
    leakage_checks["duplicate_texts_validation_vs_test"],
)
if leaked:
    raise RuntimeError("Text leakage detected between splits; review the split code.")
if leakage_checks["evaluation_languages_missing_from_train"]:
    raise RuntimeError("Some evaluation languages never appear in training.")
if leakage_checks["suspiciously_high_accuracy"]:
    print("WARNING: near-perfect accuracy is unusual; check for duplicated documents or label leakage.")

if len(misclassified):
    print("\nweakest languages by F1 among the worst 10:")
    weakest = per_language_metrics.nsmallest(10, "f1")["language"].tolist()
    display(
        results[results["expected"].isin(weakest)]
        .groupby("expected", observed=True)
        .agg(examples=("correct", "size"), accuracy=("correct", "mean"), median_length=("char_length", "median"))
        .sort_values("accuracy")
        .round(4)
    )


### 9.2 Persist evaluation results

Evaluation files are written atomically into the artifact directory created above:

- `metrics.json`: test metrics, per-language scores, length buckets, confidence summary and leakage checks.
- `confusion_matrix.csv`: full 235×235 matrix with language codes.
- `misclassified_examples.csv`: up to `CONFIG.max_misclassified_examples` errors, highest confidence first.
- `training_history.csv`: per-epoch losses, accuracies and durations, so the training curve can be reproduced without rerunning the notebook.

These files are part of the exported artifact and are hashed in the manifest.

In [ ]:
def write_text_atomic(path: Path, content: str) -> None:
    '''Write text through a temporary file and atomically replace the target.'''
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(content, encoding="utf-8")
    os.replace(temporary, path)


def write_json_atomic(path: Path, payload: object) -> None:
    '''Write JSON (sorted-stable formatting) atomically.'''
    write_text_atomic(path, json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n")


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    '''Return the SHA-256 hex digest of a file.'''
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


created_at = datetime.now(timezone.utc).isoformat()

evaluation_payload = {
    "model_version": CONFIG.model_version,
    "created_at": created_at,
    "dataset": str(CONFIG.dataset_path),
    "dataset_rows": int(len(clean_df)),
    "split_rows": split_details["rows_per_split"],
    "max_length": CONFIG.max_length,
    "vocabulary_size": len(vocabulary),
    "num_classes": len(labels),
    "best_epoch": int(best_epoch),
    "best_val_loss": float(best_val_loss),
    "best_val_accuracy": float(best_row["val_accuracy"]),
    "test": metric_summary,
    "per_language": {
        row["language"]: {
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "support": int(row["support"]),
        }
        for _, row in per_language_metrics.iterrows()
    },
    "length_buckets": length_summary.astype({"length_bucket": str}).to_dict(orient="records"),
    "confidence": confidence_summary,
    "leakage_checks": leakage_checks,
    "device": str(DEVICE),
}

write_json_atomic(ARTIFACT_DIR / "metrics.json", evaluation_payload)
write_text_atomic(
    ARTIFACT_DIR / "confusion_matrix.csv",
    pd.DataFrame(confusion, index=labels, columns=labels).to_csv(),
)
write_text_atomic(
    ARTIFACT_DIR / "misclassified_examples.csv",
    misclassified.head(CONFIG.max_misclassified_examples).to_csv(index=False),
)
write_text_atomic(ARTIFACT_DIR / "training_history.csv", history_df.to_csv(index=False))

kept_errors = min(len(misclassified), CONFIG.max_misclassified_examples)
print(f"wrote {ARTIFACT_DIR / 'metrics.json'}")
print(f"wrote {ARTIFACT_DIR / 'confusion_matrix.csv'}")
print(f"wrote {ARTIFACT_DIR / 'misclassified_examples.csv'} ({kept_errors:,} rows)")
print(f"wrote {ARTIFACT_DIR / 'training_history.csv'} ({len(history_df)} epochs)")


## 10. Export a versioned, self-contained artifact

The artifact directory (`artifacts/linguae/` by default) contains everything needed for inference:

| File | Contents |
| --- | --- |
| `model.pt` | `model.state_dict()` of the best validation checkpoint (tensors only, no pickled model) |
| `config.json` | architecture, dimensions, `max_length`, special token ids, normalisation policy |
| `vocabulary.json` | exact character→index mapping used during training |
| `labels.json` | stable `label2idx` and `idx2label` mappings |
| `metrics.json` | test metrics and evaluation metadata |
| `manifest.json` | schema/model version, timestamp, dataset path and SHA-256, training config, environment, hashes of every other artifact file |
| `linguae_inference.py` | standalone module (same source executed in section 1.1) for loading and predicting |
| `example_inference.py` | runnable loading example (`python example_inference.py`) |
| `confusion_matrix.csv`, `misclassified_examples.csv`, `training_history.csv` | evaluation and training-curve outputs from sections 8 and 9.2 |

Safeguards:

- Weights are saved with `torch.save(state_dict)`; `torch.load(..., weights_only=True)` is used on load, so loading does not execute arbitrary pickle code.
- `config.json` is sufficient to rebuild the exact architecture via `build_model`.
- All JSON and text writes go through a temporary file plus `os.replace` for atomicity.
- The notebook refuses to overwrite a non-empty artifact directory unless `CONFIG.allow_overwrite = True`; the version is recorded in `manifest.json`.
- The dataset itself is **not** copied into the artifact.
- The exported model performs one language prediction per string; it does not detect mixed-language spans.

In [ ]:
export_config = {
    "schema_version": 1,
    "model_type": "character_cnn",
    "framework": "pytorch",
    "torch_version": torch.__version__,
    "embedding_dim": int(CONFIG.embedding_dim),
    "num_filters": [int(value) for value in CONFIG.num_filters],
    "kernel_sizes": [int(value) for value in CONFIG.kernel_sizes],
    "dropout": float(CONFIG.dropout),
    "vocab_size": int(len(vocabulary)),
    "num_classes": int(len(labels)),
    "max_length": int(CONFIG.max_length),
    "padding_idx": int(PAD_ID),
    "unknown_idx": int(UNK_ID),
    "special_tokens": list(SPECIAL_TOKENS),
    "padding_side": "right",
    "pooling": "masked_max",
    "normalisation": "NFC + whitespace collapse",
    "lowercase": False,
    "prediction_scope": "single-label-per-string",
}

write_json_atomic(ARTIFACT_DIR / "config.json", export_config)
write_json_atomic(ARTIFACT_DIR / "vocabulary.json", vocabulary)
write_json_atomic(
    ARTIFACT_DIR / "labels.json",
    {
        "label2idx": label2idx,
        "idx2label": {str(index): label for index, label in idx2label.items()},
    },
)

model_path = ARTIFACT_DIR / "model.pt"
temporary_model_path = model_path.with_name(model_path.name + ".tmp")
torch.save(
    {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()},
    temporary_model_path,
)
os.replace(temporary_model_path, model_path)

write_text_atomic(ARTIFACT_DIR / "linguae_inference.py", INFERENCE_MODULE_SOURCE)

example_source = '''# Minimal standalone example for the exported Linguae artifact.
#
# Run from this directory:  python example_inference.py

from pathlib import Path

from linguae_inference import load_artifact, predict_text

ARTIFACT_DIR = Path(__file__).resolve().parent
artifact = load_artifact(ARTIFACT_DIR, device="cpu")

for text in [
    "Bonjour tout le monde, comment allez-vous ?",
    "Hello world",
    "Привет, мир",
]:
    prediction = predict_text(text, artifact)
    print(f"{prediction['language']} ({prediction['confidence']:.3f}) <- {text}")
'''
write_text_atomic(ARTIFACT_DIR / "example_inference.py", example_source)

print("artifact directory:", ARTIFACT_DIR)
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {path.name:<32} {path.stat().st_size / 1024:>9.1f} KiB")


In [ ]:
hashed_files = [
    "config.json",
    "vocabulary.json",
    "labels.json",
    "model.pt",
    "metrics.json",
    "linguae_inference.py",
    "example_inference.py",
    "confusion_matrix.csv",
    "misclassified_examples.csv",
    "training_history.csv",
]

manifest = {
    "schema_version": 1,
    "artifact_schema": "linguae.language-detector/1",
    "model_version": CONFIG.model_version,
    "created_at": created_at,
    "dataset": {
        "path": str(CONFIG.dataset_path),
        "sha256": sha256_file(CONFIG.dataset_path),
        "rows_used": int(len(clean_df)),
        "text_column": CONFIG.text_column,
        "label_column": CONFIG.label_column,
    },
    "training": {
        "seed": CONFIG.seed,
        "train_fraction": CONFIG.train_fraction,
        "val_fraction": CONFIG.val_fraction,
        "test_fraction": CONFIG.test_fraction,
        "max_length": CONFIG.max_length,
        "embedding_dim": CONFIG.embedding_dim,
        "num_filters": list(CONFIG.num_filters),
        "kernel_sizes": list(CONFIG.kernel_sizes),
        "dropout": CONFIG.dropout,
        "batch_size": CONFIG.batch_size,
        "epochs_configured": CONFIG.epochs,
        "epochs_run": int(len(history_df)),
        "best_epoch": int(best_epoch),
        "optimizer": "AdamW",
        "loss": "CrossEntropyLoss",
        "learning_rate": CONFIG.learning_rate,
        "weight_decay": CONFIG.weight_decay,
        "grad_clip_norm": CONFIG.grad_clip_norm,
        "early_stopping_patience": CONFIG.early_stopping_patience,
        "device": str(DEVICE),
        "max_rows_per_language": CONFIG.max_rows_per_language,
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "platform": platform.platform(),
    },
    "files": {name: sha256_file(ARTIFACT_DIR / name) for name in hashed_files},
    "prediction_scope": (
        "single-language classification per string; mixed-language span detection is not supported"
    ),
}

write_json_atomic(ARTIFACT_DIR / "manifest.json", manifest)
print(f"wrote {ARTIFACT_DIR / 'manifest.json'}")
print(json.dumps({"model_version": manifest["model_version"], "files": manifest["files"]}, indent=2))


In [ ]:
reloaded = load_artifact(ARTIFACT_DIR, device="cpu")

fixed_texts = test_df["text"].head(3).tolist() + [
    "Bonjour tout le monde",
    "The quick brown fox jumps over the lazy dog.",
]
fixed_sequences = [encode_text(text, vocabulary, CONFIG.max_length) for text in fixed_texts]
fixed_batch = pad_sequences(fixed_sequences, max_length=CONFIG.max_length)
model.cpu()
model.eval()

with torch.inference_mode():
    reference_probabilities = torch.softmax(model(fixed_batch), dim=1)
    reloaded_probabilities = torch.softmax(reloaded["model"](fixed_batch), dim=1)

maximum_difference = float((reference_probabilities - reloaded_probabilities).abs().max())
reference_predictions = reference_probabilities.argmax(dim=1).tolist()
reloaded_predictions = reloaded_probabilities.argmax(dim=1).tolist()

if reference_predictions != reloaded_predictions:
    raise RuntimeError(
        f"Artifact predictions differ from the trained model: "
        f"{reference_predictions} != {reloaded_predictions}"
    )
if maximum_difference > 1e-5:
    raise RuntimeError(f"Artifact probabilities differ by {maximum_difference:.2e}")

print(f"verification: {len(fixed_texts)} fixed strings, max probability difference {maximum_difference:.2e}")
print("artifact predictions match the trained model.\n")

print("example inference from the reloaded artifact:")
for text in fixed_texts[-2:] + fixed_texts[:1]:
    prediction = predict_text(text, reloaded, top_k=3)
    top_three = ", ".join(
        f"{item['language']}={item['score']:.3f}" for item in prediction["top_k"]
    )
    print(f"  {text[:70]!r} -> {prediction['language']} ({prediction['confidence']:.3f}) | {top_three}")

print("\ngenerated artifact files:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {path}")
print(f"\nstandalone module : {ARTIFACT_DIR / 'linguae_inference.py'}")
print(f"example script    : {ARTIFACT_DIR / 'example_inference.py'}")
print(f"run it with       : cd {ARTIFACT_DIR} && python example_inference.py")


## 11. Limitations and interpretation

### Dataset limitations

- WiLI-2018 paragraphs come from Wikipedia. The vocabulary, topics and register are encyclopaedic; performance on tweets, chat, code-switched text or OCR output will be lower.
- Some labels are closely related varieties (`bs`/`hr`/`sr`, `nb`/`nn`/`da`, `id`/`ms`, Arabic dialects), which are genuinely hard and often mutually intelligible.
- Wikipedia text is written, edited and spell-checked. Real-world input has typos, emoji, URLs, hashtags and mixed scripts.
- Paragraphs are long (median hundreds of characters). Short inputs such as a single word carry less evidence; section 9.1 quantifies the length effect.
- Training truncates at `CONFIG.max_length` characters. Longer documents may contain a different language later on, which this model cannot detect.

### Evaluation caveats

- Test accuracy is an average over a balanced, encyclopaedic test split. It is not a guarantee for a production traffic distribution, which is usually dominated by a few languages and contains short or noisy strings.
- Softmax scores are not calibrated probabilities. A confidence of 0.9 does not imply 90% correctness; calibration would require a separate validation protocol.
- Per-language macro-F1 is the fairer summary under imbalance, but even macro-F1 hides which specific confusions matter for an application.
- One prediction is produced per string. The model does **not** detect mixed-language spans or per-token languages, and should not be presented as doing so.
- The model is trained only on the languages present in the training split; it always returns one of those labels, even for an unsupported language or gibberish.

### Reproducibility caveats

- Seeds make splits and initialisation repeatable. GPU/MPS kernels can differ numerically, and library upgrades can change results slightly.
- The exported artifact is tied to the PyTorch version recorded in `manifest.json`/`config.json`; `state_dict` compatibility across major PyTorch versions is not guaranteed.

## 12. Summary

The notebook has produced a complete local language-identification artifact:

1. validated and profiled the dataset,
2. split it deterministically without duplicate-text leakage,
3. built a training-only character vocabulary,
4. trained a character-level CNN with early stopping,
5. evaluated once on the held-out test set with NumPy-only metrics, confusion analysis, confidence and length breakdowns,
6. exported a versioned, hash-verified artifact that ordinary Python can load with `linguae_inference.load_artifact`.

To use it in another project, copy `artifacts/linguae/` and import `linguae_inference` from that directory. To retrain a new version, change data or hyperparameters, bump `CONFIG.model_version`, and set `CONFIG.allow_overwrite` only when replacing the same version deliberately.